# Paper-style factor PCA

Run a small direct vary-one-factor PCA analysis on the smoke checkpoints, then inspect the saved results and plots. This is a descriptive diagnostic on one trained model; the small settings are only for checking the workflow. The paper analysis uses the model's full input capacity of 9 positions (BOS plus eight observations).

In [ ]:
from pathlib import Path
import json
import torch
torch.set_num_threads(1)

ROOT = Path.cwd()
if not (ROOT / 'pixi.toml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'pixi.toml').exists(), 'Open this notebook from the project workspace'
from factorization.analysis.paper import analyze_paper

RUN = ROOT / 'runs' / 'smoke'
ANALYSIS = RUN / 'analyses' / 'paper-notebook-demo-9pos'
if not (ANALYSIS / 'results.json').exists():
    analyze_paper(
        RUN, checkpoints=[0, 8], n_sequences=32, backgrounds=2,
        batch_per_background=16, max_components=6, interface='L4_mlp',
        output=ANALYSIS
    )


In [ ]:
results = json.loads((ANALYSIS / 'results.json').read_text())
print('input positions:', results['protocol']['input_length'])
print('interface:', results['protocol']['interface'])
for row in results['rows']:
    overlaps = [value[-1] for value in row['pair_overlap'].values() if value]
    print('step:', row['step'], '| combined k95:', row['k95'],
          '| mean normalized overlap at available components:',
          sum(overlaps) / len(overlaps) if overlaps else None)
figures = sorted((ANALYSIS / 'figures').glob('*.png'))
print('figures:', *figures, sep='\n')

In [ ]:
# Display saved figures without recomputing the analysis.
from IPython.display import display, Image
for path in figures:
    print(path)
    display(Image(filename=str(path)))